## 상품 CRUD - products

In [8]:
import os
from supabase import create_client
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
url = os.getenv('API_URL2')
key = os.getenv('API_KEY2')

supabase = create_client(url, key)
# print(supabase)
# print(url, key)

In [10]:
email = 'yeji@test2.org'
password = 'testtest1234'

# 회원가입
sign_in = supabase.auth.sign_up({
    'email' : email,
    'password' : password
})

In [12]:
# 로그인
log_in = supabase.auth.sign_in_with_password({
    "email": email,
    "password": password
})

# log_in.user

In [15]:
# # 로그인 한 유저 확인
# login_user = supabase.auth.get_user()

# if login_user :
#     print("로그인 상태 :", login_user)
# else : 
#     print("미로그인 상태")

In [16]:
class product_service: # create, update, delete 등등
    
    def __init__ (self) : 
        user_info = supabase.auth.get_user()

        if not user_info : 
            raise PermissionError("로그인 후 다시 시도하세요")

        self.user = user_info.user


    #~~[create] - supabase table editor로 products 테이블 구조 만듦~~


    # [insert] - 신규 상품등록
    def new_product(self, name, price, description) :
        
        # 1. 현재 로그인한 사용자의 회원 유형 조회
        user_detail = (
            supabase.table("user_details")
                .select("type")
                .eq("id", self.user.id)
                .execute()
        )

        # 2. 회원 유형이 seller인 경우만 상품 등록할 수 있도록 
        if not user_detail.data : 
            raise PermissionError("회원가입을 먼저 진행해주세요")

        if user_detail.data["type"] != "seller" :
            raise PermissionError("Seller로 등록된 회원이 아닙니다")
        
        # 3. seller 회원 - 상품등록
        new_product = (
            supabase.table('products')
                .insert({
                    "name" : name,
                    "price" : price,
                    "description" : description,
                    "seller_id" : self.user.id
                })
                .execute()
        )
        return f"등록한 데이터 : {new_product.data}"

    
    # [select] - 상품 조회
    def get_product(self, name) :
        get_product = (
            supabase.table("products")
                .select("name", "price", "description")
                .ilke("name", f"%{name}%")
                .execute()
        )

        return f"조회된 데이터 : {get_product.data}"


    # [update] - 상품명 수정
    def modify_product_name(self, id, name) :
        modify_product_name = (
            supabase.table("products")
            .update({
                "name" : name
                })
                .eq("id", id)
                .eq("seller_id", self.user.id) # seller만 수정 가능하도록
                .execute()
        )

        return f"수정된 [[상품명]] 데이터 : {modify_product_name.data}"


    # [update] - 상품가격 수정
    def modify_product_price(self, id, price) : 
        modify_product_price = (
            supabase.table("products")
            .update({
                "price" : price
            })
            .eq("id", id)
            .eq("seller_id", self.user.id) # seller만 수정 가능하도록
            .execute()
        )

        return f"수정된 [[가격]] 데이터 : {modify_product_price.data}"


     # [update] - 상품 설명 수정
    def modify_product_description(self, id, description) :
        modify_product_description = (
            supabase.table("products")
            .update({
                 "description" : description
            })
            .eq("id", id)
            .eq("seller_id", self.user.id) # seller만 수정 가능하도록
            .execute()
        )

        return f"수정된 [[상품설명]] 데이터 : {modify_product_description.data}"

    # [delete] - 상품 삭제
    def delete_product (self, id, name) :
        delete_product = (
            supabase.table("products")
            .delete({
                "name" : name
            })
            .eq("id", id)
            .eq("seller_id", self.user.id)
            .execute()
        )

        return f"삭제된 [[상품명]] 데이터 : {delete_product}"